In [1]:
import json
from io import BytesIO

import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from pyarrow import fs
from PIL import Image
import boto3

In [2]:

# Read the AWS Region used by the current Boto3 session.
AWS_REGION = boto3.Session().region_name

print(AWS_REGION)

# Create the PyArrow filesystem used to communicate with Amazon S3.
s3 = fs.S3FileSystem(region=AWS_REGION)

# Define the S3 bucket name.
BUCKET_NAME = "mushroom-ml-mikael-2026-000"

# Define the input dataset paths.
TRAIN_PATH = (
    f"{BUCKET_NAME}/processed/parquet/"
    "train_balanced_augmented.parquet"
)

VALIDATION_PATH = (
    f"{BUCKET_NAME}/processed/parquet/"
    "validation.parquet"
)

TEST_PATH = (
    f"{BUCKET_NAME}/processed/parquet/"
    "test.parquet"
)

# Define the output path for the label mapping.
LABEL_MAPPING_PATH = (
    f"{BUCKET_NAME}/training/input/"
    "label_mapping.json"
)

us-east-1


In [3]:
# Read the training dataset from S3.
train_table = pq.read_table(
    TRAIN_PATH,
    filesystem=s3
)

# Read the validation dataset from S3.
validation_table = pq.read_table(
    VALIDATION_PATH,
    filesystem=s3
)

# Read the test dataset from S3.
test_table = pq.read_table(
    TEST_PATH,
    filesystem=s3
)

# The natural dataset was initially split into 70% training, 15% validation, and 15% test. 
# After balancing the training split through data augmentation, 
# the final dataset contains 2,448 training images, 365 validation images, and 366 test images.

print("Train rows:", train_table.num_rows)
print("Validation rows:", validation_table.num_rows)
print("Test rows:", test_table.num_rows)

Train rows: 2448
Validation rows: 365
Test rows: 366


In [4]:
# Display the schema of each dataset.
print("Train columns:", train_table.column_names)
print("Validation columns:", validation_table.column_names)
print("Test columns:", test_table.column_names)

Train columns: ['image', 'label', 'bbox', 'mushroom_name', 'edibility', 'entry_id', 'origin']
Validation columns: ['image', 'label', 'bbox', 'split', 'mushroom_name', 'edibility', 'entry_id']
Test columns: ['image', 'label', 'bbox', 'split', 'mushroom_name', 'edibility', 'entry_id']


In [5]:
# Define the columns required by the training pipeline.
required_columns = {
    "image",
    "mushroom_name",
    "edibility",
    "entry_id"
}

# Check whether every dataset contains the required columns.
train_has_required_columns = required_columns.issubset(
    set(train_table.column_names)
)

validation_has_required_columns = required_columns.issubset(
    set(validation_table.column_names)
)

test_has_required_columns = required_columns.issubset(
    set(test_table.column_names)
)

print("Train has required columns:", train_has_required_columns)
print("Validation has required columns:", validation_has_required_columns)
print("Test has required columns:", test_has_required_columns)

Train has required columns: True
Validation has required columns: True
Test has required columns: True
